In [ ]:
import json
from pathlib import Path
import pandas as pd

# Load cleaned json file from data/interim
data_path = Path("../data/interim/cleaned_details.json")

records = json.loads(data_path.read_text(encoding="utf-8"))
print(f"Total records: {len(records)}")

: 

In [ ]:
df = pd.DataFrame([
    {
        "title":            r.get("title"),
        "series_name":      r.get("series_name"),
        "episode_number":   r.get("episode_number"),
        "description":      r.get("description"),
        "duration_minutes": r.get("duration_minutes"),
        "release_date":     r.get("release_date"),
        "label":            r.get("label"),
        "cover_url":        r.get("cover_url"),
        "order_number":     r.get("order_number"),
        "source_url":       r.get("source_url"),
        "n_speakers":       len(r.get("speakers") or []),
        "n_genres":         len(r.get("genres") or []),
    }
    for r in records
])

print(f"Shape: {df.shape}")

In [ ]:
with_description = sum(1 for r in records if r.get("description"))
print(f"With description:    {with_description}")
print(f"Without description: {len(records) - with_description}")
print(f"Percentage:          {with_description / len(records) * 100:.1f}%")
df["description"].notna().sum()

In [ ]:
# Data types and memory usage
df.info()

In [ ]:
df.head()

In [ ]:
# Missing values — how complete is the data?
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({"missing": missing, "percent": missing_pct})

In [ ]:
# Episodes without a title — should be 0, otherwise parser issue
no_title = df[df["title"].isnull()]
print(f"Episodes without title: {len(no_title)}")
if len(no_title) > 0:
    print(no_title[["source_url"]].head(10))

In [ ]:
# Series: count and episodes per series
series_counts = df.groupby("series_name").size().sort_values(ascending=False)
print(f"Number of series: {len(series_counts)}\n")
series_counts.head(20)

In [ ]:
# Parse release date and check for problems
df["release_date_parsed"] = pd.to_datetime(
    df["release_date"], format="%d.%m.%Y", errors="coerce"
)

n_missing = df["release_date"].isnull().sum()
n_failed  = df["release_date_parsed"].isnull().sum()
n_errors  = n_failed - n_missing

print(f"No date at all:       {n_missing}")
print(f"Failed to parse:      {n_errors}  ← cleaning candidates")
print(f"Successfully parsed:  {len(df) - n_failed}")

# Which values failed?
df[df["release_date"].notnull() & df["release_date_parsed"].isnull()][
    ["title", "release_date", "source_url"]
].head(20)

In [ ]:
# Releases per year
df["year"] = df["release_date_parsed"].dt.year
df["year"].value_counts().sort_index()

In [ ]:
# Outliers in duration
print(df["duration_minutes"].describe())

print("\nVery short (< 5 min):")
print(df[df["duration_minutes"] < 5][["title", "series_name", "duration_minutes"]].head(10))

print("\nVery long (> 180 min):")
print(df[df["duration_minutes"] > 180][["title", "series_name", "duration_minutes"]].head(10))

In [ ]:
# Explore genres and count
genres_series = pd.Series([
    g for r in records for g in (r.get("genres") or [])
])
print(f"Unique genres: {genres_series.nunique()}\n")
genres_series.value_counts().head(20)

In [ ]:
with_genres = sum(1 for r in records if r.get("genres"))
print(f"Records with genres: {with_genres}")
print(f"Total genre assignments: {sum(len(r.get('genres') or []) for r in records)}")

In [ ]:
# Most active voice actors
all_speakers = pd.Series([
    s["speaker"] for r in records for s in (r.get("speakers") or [])
])
print(f"Total speaker credits: {len(all_speakers)}")
print(f"Unique speakers:       {all_speakers.nunique()}\n")
all_speakers.value_counts().head(20)

In [ ]:
# Normalize speaker names for comparison
# Format is "Lastname, Firstname" — let's check for umlaut variants
speakers_df = pd.DataFrame([
    {
        "speaker": s["speaker"],
        "role":    s["role"],
        "title":   r.get("title"),
        "series":  r.get("series_name"),
    }
    for r in records
    for s in (r.get("speakers") or [])
])

print(f"Total speaker credits: {len(speakers_df)}")
print(f"Unique speaker names:  {speakers_df['speaker'].nunique()}")
speakers_df.head(5)

In [ ]:
# Look for potential umlaut variants of the same person
# Strategy: normalize umlauts and compare against original
umlaut_map = str.maketrans({
    "ä": "ae", "ö": "oe", "ü": "ue",
    "Ä": "Ae", "Ö": "Oe", "Ü": "Ue",
    "ß": "ss",
})

def normalize_name(name: str) -> str:
    return name.translate(umlaut_map).lower().strip()

speakers_df["speaker_normalized"] = speakers_df["speaker"].map(normalize_name)

# Group by normalized name — if a normalized name has more than one
# original spelling, that's a candidate for deduplication
variants = (
    speakers_df.groupby("speaker_normalized")["speaker"]
    .nunique()
    .reset_index()
    .rename(columns={"speaker": "n_variants"})
)

suspicious = variants[variants["n_variants"] > 1].sort_values("n_variants", ascending=False)
print(f"Normalized names with multiple spellings: {len(suspicious)}")
suspicious.head(20)

In [ ]:
# Inspect the actual variants for each suspicious name
for norm_name in suspicious["speaker_normalized"].head(20):
    original_variants = (
        speakers_df[speakers_df["speaker_normalized"] == norm_name]["speaker"]
        .value_counts()
    )
    print(f"\n{norm_name}:")
    print(original_variants.to_string())

In [ ]:
# Check for other common inconsistencies:
# trailing/leading whitespace, double spaces, etc.
has_whitespace_issue = speakers_df["speaker"].str.contains(r"^\s|\s$|\s{2,}", regex=True)
print(f"Names with whitespace issues: {has_whitespace_issue.sum()}")
speakers_df[has_whitespace_issue][["speaker"]].drop_duplicates().head(20)

In [ ]:
# Same check for roles
roles_df = speakers_df["role"].value_counts()
print(f"Unique roles: {len(roles_df)}\n")

# Roles that look like they might be the same
# e.g. "Erzähler" vs "Erzaehler" vs "erzähler"
roles_normalized = pd.Series(
    speakers_df["role"].map(normalize_name).value_counts()
)

role_variants = (
    speakers_df.groupby(speakers_df["role"].map(normalize_name))["role"]
    .nunique()
    .reset_index(name="n_variants")  # direkt den neuen Spaltennamen setzen
)

suspicious_roles = role_variants[role_variants["n_variants"] > 1]
print(f"Roles with multiple spellings: {len(suspicious_roles)}")
suspicious_roles.head(20)

In [ ]:
# Inspect the actual variants for each suspicious role
for norm_role in suspicious_roles["role"].head(20):
    original_variants = (
        speakers_df[speakers_df["role"].map(normalize_name) == norm_role]["role"]
        .value_counts()
    )
    print(f"\n{norm_role}:")
    print(original_variants.to_string())

In [ ]:
# How often does each role appear? Helps spot noise vs real roles
# e.g. one-off typos will have very low counts
role_counts = speakers_df["role"].value_counts()
print("Most common roles:")
print(role_counts.head(20))
print("\nRoles that appear only once (potential noise):")
print(role_counts[role_counts == 1].head(20))

In [ ]:
# Summary: what needs fixing before writing to DB?
print("=== Data Quality Summary ===\n")
print(f"Speaker name variants (umlaut issues): {len(suspicious)}")
print(f"Whitespace issues in names:            {has_whitespace_issue.sum()}")
print(f"Role spelling variants:                {len(suspicious_roles)}")
print(f"\nRecords with no speakers at all:       {(df['n_speakers'] == 0).sum()}")
print(f"Records with no genres:                {(df['n_genres'] == 0).sum()}")
print(f"Records with no release date:          {df['release_date'].isnull().sum()}")
print(f"Records with no description:           {df['description'].isnull().sum()}")